# Regression
### Today I want to practice on a REGRESSION PROJECT!

For this project, we have prepared the information of almost 4000 apartments in Tehran. All data is completely real. Our task is to estimate the price in dollars or tomans using the features of the dataset. The data is stored in the housePrice.csv file.
- House size in meters (Area)
- Number of bedrooms
- Is there a parking lot or not?
- Does it have a warehouse or not?
- Does it have an elevator or not?
- An approximate address in Tehran (Address)
- Price in Tomans
- Price in dollars (Price(USD))

In this dataset, some houses do not have addresses, and also the size of some houses is entered incorrectly (they have a very large value). For this purpose, we must also manage these items and remove them from our dataset.

In [ ]:
pip install seaborn

#### Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, normalize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import warnings
warnings.filterwarnings('ignore')

#### Now we want to Load the dataset

In [ ]:
df = pd.read_csv('1632300362534233.csv')

In [ ]:
df.info()

#### Display basic information

In [ ]:
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
df["Area"] = pd.to_numeric(df["Area"], errors='coerce')
for d in df["Area"]:
    if d > 300:
        print(d)

#### Clean up excess data
Create a copy for cleaning

In [ ]:
df_clean = df.copy()

Remove houses without address

In [ ]:
df_clean = df_clean.dropna(subset=['Address'])

Remove extreme outliers in Area (assuming reasonable range is 20-500 sqm)

In [ ]:
df_clean = df.copy()
df_clean = df_clean.dropna(subset=['Address'])
df_clean['Area'] = pd.to_numeric(df_clean['Area'], errors='coerce')

df_clean = df_clean[(df_clean['Area'] >= 20) & (df_clean['Area'] <= 500)]
# df_clean = df_clean[(df_clean['Price(USD)'] >= 20) & (df_clean['Price(USD)'] <= 100000)]
print(df_clean.shape)

print(f"Original dataset size: {df.shape}")
print(f"Cleaned dataset size: {df_clean.shape}")
print(f"Number of rows removed: {df.shape[0] - df_clean.shape[0]}")

In [ ]:
df.nunique()

#### Perform further exploratory analysis to gain better insights from the data

 Check correlation between features


In [ ]:
from sklearn.preprocessing import LabelEncoder

df_encode = df_clean.copy()
le = LabelEncoder()
df_encode['Address'] = le.fit_transform(df_encode['Address'])
df_encode['Parking'] = df_encode['Parking'].astype(int)
df_encode['Warehouse'] = df_encode['Warehouse'].astype(int)
df_encode['Elevator'] = df_encode['Elevator'].astype(int)
df_encode

numeric_features = ['Area', 'Room', 'Parking', 'Warehouse', 'Elevator', 'Address', 'Price', 'Price(USD)']

plt.figure(figsize=(12, 10))
correlation_matrix = df_encode[numeric_features].corr()

sns.heatmap(correlation_matrix, annot=True, cmap='RdYlGn', center=0.5, 
            fmt='.2f', linewidths=0.5, square=True, cbar_kws={"shrink": 0.8})

plt.title('Correlation Matrix')
plt.xticks(rotation=45 , ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


Check distribution of categorical features

In [ ]:
area_data = df_clean['Area'].values.reshape(-1, 1)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_clean['Area_Cluster'] = kmeans.fit_predict(area_data)

area_ranges = []
for i in range(5):
    cluster_values = df_clean[df_clean['Area_Cluster'] == i]['Area']
    min_area = cluster_values.min()
    max_area = cluster_values.max()
    area_ranges.append(f"{min_area:.0f}-{max_area:.0f}m²")
    print(f"category {i}: {min_area:.0f} - {max_area:.0f} m² ")

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

sns.countplot(data=df_clean, x='Room', ax=axes[0, 0])
sns.countplot(data=df_clean, x='Parking', ax=axes[0, 1])
sns.countplot(data=df_clean, x='Warehouse', ax=axes[1, 0])
sns.countplot(data=df_clean, x='Elevator', ax=axes[1, 1])

sns.countplot(data=df_clean, x='Area_Cluster', ax=axes[2, 0], 
              order=sorted(df_clean['Area_Cluster'].unique()))

labels = [f'category {i}\n({area_ranges[i]})' for i in sorted(df_clean['Area_Cluster'].unique())]
axes[2, 0].set_xticklabels(labels)

sns.boxplot(data=df_clean, x='Area_Cluster', y='Price(USD)', ax=axes[2, 1],
           order=sorted(df_clean['Area_Cluster'].unique()))
axes[2, 1].set_xlabel('Area category')
axes[2, 1].set_ylabel('price(USD)')
axes[2, 1].set_xticklabels([f'category {i}' for i in sorted(df_clean['Area_Cluster'].unique())])

plt.tight_layout()
plt.show()

print("\nAverage price per area category: ")
price_by_cluster = df_clean.groupby('Area_Cluster')['Price(USD)'].mean().sort_index()
for cluster, avg_price in price_by_cluster.items():
    print(f"category {cluster}: ${avg_price:,.0f}")

##### Analyze price distribution by district (top 20 districts)

In [ ]:
top_districts = df_clean['Address'].value_counts().head(20).index
plt.figure(figsize=(15, 8))
sns.boxplot(data=df_clean[df_clean['Address'].isin(top_districts)], 
            x='Address', y='Price(USD)')
plt.xticks(rotation=45)
plt.title('Price Distribution by District (Top 20)')
plt.show()

#### Data preparation for modeling. Using Price(USD) as a target because it has a better scale for modeling

In [ ]:
X = df_clean.drop(['Price', 'Price(USD)'], axis=1)
y = df_clean['Price(USD)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['Area', 'Room']
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())])

categorical_features = ['Parking', 'Warehouse', 'Elevator', 'Address']
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

#### Train by Linear Regression

In [ ]:
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', LinearRegression())])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model Evaluation:")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price (USD)')
plt.ylabel('Predicted Price (USD)')
plt.title('Actual vs Predicted Prices')
plt.show()

#### Train by Gradient Boosting Regression

In [ ]:
model = Pipeline(steps=[('preprocessor', preprocessor),
                        ('regressor', GradientBoostingRegressor(random_state=4))])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Model Evaluation:")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")

plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price (USD)')
plt.ylabel('Predicted Price (USD)')
plt.title('Actual vs Predicted Prices')
plt.show()

#### Examining the importance of features

Get feature importance

In [ ]:
feature_names = numeric_features + list(model.named_steps['preprocessor']
                                      .named_transformers_['cat']
                                      .named_steps['onehot']
                                      .get_feature_names_out(categorical_features))
print(vars(model))
importances = model.named_steps['regressor'].feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(12, 8))
feat_imp.head(20).plot(kind='barh')
plt.title('Top 20 Feature Importances')
plt.xlabel('Importance')
plt.show()

feature_names = numeric_features + list(model.named_steps['preprocessor']
                                      .named_transformers_['cat']
                                      .named_steps['onehot']
                                      .get_feature_names_out(categorical_features))

In [ ]:
X = df_encode.drop(['Price', 'Price(USD)'], axis=1)
y = df_encode['Price(USD)']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['Area', 'Room', 'Address']
categorical_features = ['Parking', 'Warehouse', 'Elevator']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ])

#### Definition of models and preparation

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'SVR': SVR(kernel='rbf', C=1.0, gamma='scale')
}

results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # save results
    results[name] = {
        'model': pipeline,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'predictions': y_pred
    }
    
    print(f"{name}:")
    print(f"  MAE: {mae:.2f}")
    print(f"  MSE: {mse:.2f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  R2 Score: {r2:.4f}")
    print("-" * 50)

#### Comparison of models

In [ ]:
print("\n" + "="*60)
print("Comparison of performance of models:")
print("="*60)

comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE': [results[model]['MAE'] for model in results],
    'RMSE': [results[model]['RMSE'] for model in results],
    'R2 Score': [results[model]['R2'] for model in results]
})

print(comparison_df.sort_values('R2 Score', ascending=False))

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

models_names = list(results.keys())
r2_scores = [results[model]['R2'] for model in models_names]

axes[0, 0].barh(models_names, r2_scores, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[0, 0].set_title('Comparison of R² Score models')
axes[0, 0].set_xlabel('R² Score')
axes[0, 0].set_xlim(0, 1)

rmse_scores = [results[model]['RMSE'] for model in models_names]
axes[0, 1].barh(models_names, rmse_scores, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[0, 1].set_title('Comparison of RMSE models')
axes[0, 1].set_xlabel('RMSE')

# Plot of predictions vs actual values ​​for the best model
best_model_name = max(results, key=lambda x: results[x]['R2'])
best_model = results[best_model_name]['model']
y_pred_best = results[best_model_name]['predictions']

axes[1, 0].scatter(y_test, y_pred_best, alpha=0.6)
axes[1, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1, 0].set_xlabel('Actual values')
axes[1, 0].set_ylabel('Predictions')
axes[1, 0].set_title(f'Predictions vs Actual values - {best_model_name}')

# Remainders plot for the best model
residuals = y_test - y_pred_best
axes[1, 1].scatter(y_pred_best, residuals, alpha=0.6)
axes[1, 1].axhline(y=0, color='r', linestyle='--')
axes[1, 1].set_xlabel('Predictions')
axes[1, 1].set_ylabel('Remainders')
axes[1, 1].set_title(f'Remainders - {best_model_name}')

plt.tight_layout()
plt.show()

#### Cross-Validation for more certainty

In [ ]:
print("\n" + "="*60)
print("Evaluation Cross-Validation (5-fold):")
print("="*60)

cv_results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
    cv_results[name] = {
        'Mean R2': cv_scores.mean(),
        'Std R2': cv_scores.std(),
        'All Scores': cv_scores
    }
    
    print(f"{name}:")
    print(f"  Mean R2: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    print(f"  Scores: {cv_scores}")

#### Feature importance analysis for Random Forest

In [ ]:
print("\n" + "="*60)
print("Feature importance analysis for Random Forest:")
print("="*60)

rf_model = results['Random Forest']['model']
feature_names = (numeric_features + 
                list(rf_model.named_steps['preprocessor']
                    .named_transformers_['cat']
                    .get_feature_names_out(categorical_features)))

importances = rf_model.named_steps['regressor'].feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print(feature_importance_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_df, x='Importance', y='Feature')
plt.title('Feature importance in Random Forest model')
plt.tight_layout()
plt.show()

#### Choosing the best model

In [ ]:
best_model_name = max(results, key=lambda x: results[x]['R2'])
best_model = results[best_model_name]['model']

print(f"\n The best model is: {best_model_name}")
print(f" R² Score: {results[best_model_name]['R2']:.4f}")
print(f" RMSE: {results[best_model_name]['RMSE']:.2f}")

# predict the best mosel
sample_prediction = best_model.predict(X_test.head(1))
print(f"\n Sample prediction:")
print(f"Actual values: {y_test.head(1).values[0]:.2f}")
print(f"Model prediction: {sample_prediction[0]:.2f}")